In [1]:
from get_terms import TermCollector
tc = TermCollector("/home/jiaruil5/multilingual/multilingual-model-card/src/dictionary_collection/growing_dict/mturk.json")
tc.get_occurrences("Hi, this is Elena and I'm going to be presenting our work, Detecting Unassimilated Borrowings in Spanish: An Annotated Corpus and Approaches to Modeling.\n")

(119, 125) corpus corpus
(119, 125) corpus corpus
(109, 125) annotated corpus annotated corpus


{'Corpora': {'occurrences': ['Corpus'],
  'English': 'Corpora',
  'context': '1: <mark>Corpora</mark> are split into a 60/40 train/test split. Selected target identities and the size of each corpus can be found in Table 2. These identity-specific corpora, which are samples of existing publicly available datasets, are available at https://osf.io/53tfs/.<br>',
  'Arabic': 'المواد اللغوية',
  'Chinese': '语料库',
  'French': 'Corpus',
  'Japanese': 'コーパス',
  'Russian': 'Корпусы'},
 'Corpus': {'occurrences': ['Corpus'],
  'English': 'Corpus',
  'context': '1: <mark>Corpus</mark> Heterogeneity. Heterogeneous datasets form an obstacle for profound linguistic tools such as syntactic or dependency parsers, since they commonly work well when trained and applied to a specific domain, but are prone to produce incorrect results when used in a different genre of text.<br>2: In this paper, we address this gap. Our work has three parts. (i) <mark>Corpus</mark> collection. We collect Glot2000-c, a corpus

In [1]:
from fuzzywuzzy import fuzz
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

/data/user_data/jiaruil5/miniconda3/envs/new/lib/python3.11/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [85]:
def preprocess_text(text):
    lemmatizer = WordNetLemmatizer()
    tokens = word_tokenize(text.lower())
    print(tokens)
    token_indices = []
    current_position = 0
    for token in tokens:
        start_idx = text.lower().find(token, current_position)
        end_idx = start_idx + len(token)
        token_indices.append((token, start_idx, end_idx))
        current_position = end_idx  # Update current position for the next search

    lemmatized_tokens = [lemmatizer.lemmatize(token) for token, _, _ in token_indices]
    return lemmatized_tokens, token_indices

def get_substrings(words, n):
    substrings = []
    for i in range(len(words) - n + 1):
        substring = " ".join(words[i:i+n])
        substrings.append(substring)
    return substrings

def get_original_indices(lemmatized_tokens, token_indices, substring):
    substring = substring.split()
    for i in range(len(token_indices) - len(substring) + 1):
        # Check if the n adjacent lemmatized tokens match
        match = True
        for j in range(len(substring)):
            if lemmatized_tokens[i + j] != substring[j]:
                match = False
                break
        if match:
            # Get the original start and end indices of the matched sequence
            start_idx = token_indices[i][1]
            end_idx = token_indices[i + len(substring) - 1][2]
            return start_idx, end_idx
    return None

def find_terminology(in_paragraph, terminology):
    paragraph = in_paragraph
    paragraph, indices = preprocess_text(paragraph)
    terminology_lst, _ = preprocess_text(terminology)
    terminology = " ".join(terminology_lst)
    token_counts_paragraph = len(paragraph)
    token_counts = len(terminology_lst)
    substrings = get_substrings(paragraph, token_counts)
    if token_counts < token_counts_paragraph:
        substrings += get_substrings(paragraph, token_counts + 1)
    if token_counts > 0:
        substrings += get_substrings(paragraph, token_counts - 1)
    results = []
    for substring in substrings:
        if normalized_levenshtein_similarity(terminology, substring) > 0.9:
            sub_indices = get_original_indices(paragraph, indices, substring)
            print(sub_indices, terminology, substring)
            if sub_indices is not None:
                results.append(in_paragraph[sub_indices[0]:sub_indices[1]])
    return results

In [86]:
find_terminology(
    "1: For all the methods, we used <mark>10-fold cross validation</mark> (i.e., each fold we have 556 training and 62 test samples) to tune free parameters, e.g., the kernel form and parameters for GPOR and LapSVM. Note that all the alternative methods stack X and Z together into a whole data matrix and ignore their heterogeneous nature.<br>2: Features associated one-to-one with a vertical (Clarity, ReDDE, the query likelihood given the vertical's query-log and Soft.ReDDE) were normalized across verticals before scaling. Supervised training/testing was done via <mark>10-fold cross validation</mark>. Parameter τ was tuned for each training fold on the same 500 query validation set used for our single feature baselines formulation.",
    "formulation"
)

['1', ':', 'for', 'all', 'the', 'methods', ',', 'we', 'used', '<', 'mark', '>', '10-fold', 'cross', 'validation', '<', '/mark', '>', '(', 'i.e.', ',', 'each', 'fold', 'we', 'have', '556', 'training', 'and', '62', 'test', 'samples', ')', 'to', 'tune', 'free', 'parameters', ',', 'e.g.', ',', 'the', 'kernel', 'form', 'and', 'parameters', 'for', 'gpor', 'and', 'lapsvm', '.', 'note', 'that', 'all', 'the', 'alternative', 'methods', 'stack', 'x', 'and', 'z', 'together', 'into', 'a', 'whole', 'data', 'matrix', 'and', 'ignore', 'their', 'heterogeneous', 'nature.', '<', 'br', '>', '2', ':', 'features', 'associated', 'one-to-one', 'with', 'a', 'vertical', '(', 'clarity', ',', 'redde', ',', 'the', 'query', 'likelihood', 'given', 'the', 'vertical', "'s", 'query-log', 'and', 'soft.redde', ')', 'were', 'normalized', 'across', 'verticals', 'before', 'scaling', '.', 'supervised', 'training/testing', 'was', 'done', 'via', '<', 'mark', '>', '10-fold', 'cross', 'validation', '<', '/mark', '>', '.', 'param

['formulation']

In [50]:
!pip install python-Levenshtein

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.2/177.2 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 62.0 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: rapidfuzz
    Found existing installation: rapidfuzz 3.4.0
    Uninstalling rapidfuzz-3.4.0:
      Successfully uninstalled rapidfuzz-3.4.0


In [57]:
from Levenshtein import distance

def normalized_levenshtein_similarity(str1, str2):
    # Calculate Levenshtein distance
    dist = distance(str1, str2)
    # Normalize by the length of the longer string
    max_len = max(len(str1), len(str2))
    return 1 - dist / max_len if max_len else 1.0

# Example usage
similarity = normalized_levenshtein_similarity("asdsgsdfegdj", "asdsgsdfegdse")
print(f"Normalized Levenshtein Similarity: {similarity}")


Normalized Levenshtein Similarity: 0.8461538461538461


In [9]:
def get_adjacent_substrings(sentence, n):
    words = sentence.split()  # Split the sentence into words
    substrings = []

    # Loop through the sentence to get substrings of n adjacent words
    for i in range(len(words) - n + 1):
        substring = ' '.join(words[i:i + n])  # Join n adjacent words with spaces
        substrings.append(substring)
    
    return substrings

# Example usage:
sentence = "This is an example sentence for generating adjacent substrings"
n = 3
result = get_adjacent_substrings(sentence, n)
print(result)


['This is an', 'is an example', 'an example sentence', 'example sentence for', 'sentence for generating', 'for generating adjacent', 'generating adjacent substrings']


In [10]:
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

def preprocess_text_with_indices(text):
    lemmatizer = WordNetLemmatizer()
    tokens = word_tokenize(text.lower())

    # Store original indices of each token
    token_indices = []
    current_position = 0
    for token in tokens:
        start_idx = text.lower().find(token, current_position)
        end_idx = start_idx + len(token)
        token_indices.append((token, start_idx, end_idx))
        current_position = end_idx  # Update current position for the next search

    # Lemmatize tokens and map to original indices
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token, _, _ in token_indices]

    # Store lemmatized tokens with their original indices
    lemmatized_token_indices = list(zip(lemmatized_tokens, token_indices))

    return lemmatized_token_indices

def get_original_indices(text, n_adjacent_tokens):
    lemmatized_token_indices = preprocess_text_with_indices(text)

    # Split the adjacent lemmatized tokens into individual tokens
    n_adjacent_tokens_split = n_adjacent_tokens.split()

    # Search for a matching sequence of lemmatized tokens in the original indices list
    for i in range(len(lemmatized_token_indices) - len(n_adjacent_tokens_split) + 1):
        # Check if the n adjacent lemmatized tokens match
        match = True
        for j in range(len(n_adjacent_tokens_split)):
            if lemmatized_token_indices[i + j][0] != n_adjacent_tokens_split[j]:
                match = False
                break
        if match:
            # Get the original start and end indices of the matched sequence
            start_idx = lemmatized_token_indices[i][1][1]
            end_idx = lemmatized_token_indices[i + len(n_adjacent_tokens_split) - 1][1][2]
            return start_idx, end_idx

    # Return None if no match is found
    return None

# Example usage
text = "Cats are running faster than ever."
n_adjacent_tokens = "cat are running"  # Example of n adjacent lemmatized tokens
result = get_original_indices(text, n_adjacent_tokens)
print(result)


(0, 16)
